In [0]:
import json
import requests
from datetime import datetime
from pyspark.sql.types import StructType, StructField, StringType

# ✅ Get context from job
ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
run_id = ctx.runId().get()  
host = ctx.apiUrl().get()
token = ctx.apiToken().get()

# ✅ API Call to get run details
headers = {"Authorization": f"Bearer {token}"}
url = f"{host}/api/2.1/jobs/runs/get?run_id={run_id}"
response = requests.get(url, headers=headers)
run_data = response.json()

# ✅ Extract IDs
job_id = run_data.get("job_id")
job_run_id = run_data.get("run_id")
task_run_id = run_data.get("tasks", [{}])[0].get("run_id")  # Assumes at least one task

# ✅ Extract and convert timestamps
start_ts = run_data.get("start_time")
end_ts = run_data.get("end_time")

start_time_str = datetime.fromtimestamp(start_ts / 1000.0).strftime("%Y-%m-%d %H:%M:%S") if start_ts else None
end_time_str = datetime.fromtimestamp(end_ts / 1000.0).strftime("%Y-%m-%d %H:%M:%S") if end_ts else None

# ✅ Print for confirmation
print(f"✅ Job ID: {job_id}")
print(f"✅ Job Run ID: {job_run_id}")
print(f"✅ Task Run ID: {task_run_id}")
print(f"✅ Start Time: {start_time_str}")
print(f"✅ End Time: {end_time_str}")

# ✅ Define schema
schema = StructType([
    StructField("Start_Load_Date", StringType(), True),
    StructField("End_Load_Date", StringType(), True)
])

# ✅ Write to your Delta table
df = spark.createDataFrame([(start_time_str, end_time_str)],["Start_Load_Date", "End_Load_Date"])
df.write.format("delta").mode("append").saveAsTable("oh_apm_stg.vendor_extracts.load_report_log_cpc_Stg_Ref")
